In [2]:
import sys
sys.path.insert(0, "/home/s232534/scripts")

import re
from pathlib import Path
#from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import geopandas as gpd
import numpy as np
from tqdm import tqdm

import signal_segmentation
from map_matching import routes
from typing import Dict

import trip_classification # handy functions to run clean XGBoost classifiers

import ast

from exploration import functions
from optimal_percentiles import percentile_calculate
from gtfs_functions import calculate_optimal_p_stops

from sklearn.metrics import classification_report, accuracy_score, f1_score, make_scorer

from xgboost import XGBClassifier
import joblib


# Verify segment data

In [29]:
# --- Load dfs ---
df_segments = pd.read_csv("/home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt.csv")
df_trips_dataset = pd.read_csv("/home/lauale/netmob2025/data/trips_dataset.csv")

# Convert the string lists into actual object as type list.
df_segments["SEGMENTS_BOUNDS"] = df_segments["SEGMENTS_BOUNDS"].apply(ast.literal_eval)

In [4]:
df_segments["SEGMENTS_BOUNDS"][0]

[50, 130, 170, 211]

# Building single-modal XGBoost classifier

Filtration of the single-modal dataset like everytime removing outliers essentially. End goal = features for single-modal model 

In [13]:
df_stats = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_percentile_map = pd.read_parquet("/home/s232534/data/percentile_pipeline/percentiles_071125.parquet")
df_stops = pd.read_parquet("/home/s232534/data/stops_analysis_071125/stops_statistical_summary_enriched.parquet")

# Uni-modal ONLY.
df_stats = df_stats.loc[~df_stats['MULTI_MODAL']] 
df_percentile_map = df_percentile_map.loc[~df_percentile_map['MULTI_MODAL']] 
df_stops = df_stops.loc[~df_stops['MULTI_MODAL']] 

# ---
# Drop modes we will not be using for this thesis and modes to merge and remap:
# ---
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

df_stats = functions.remap_and_drop_modes_df(df_stats, mode_remap, modes_to_drop)
df_percentile_map = functions.remap_and_drop_modes_df(df_percentile_map, mode_remap, modes_to_drop)
df_stops = functions.remap_and_drop_modes_df(df_stops, mode_remap, modes_to_drop)

# ---
# Filter speeds
# ---
threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_stats = functions.filter_motion_data_speed_avg(df_stats, threshold)
df_stats = functions.filter_temporal_data(df_stats, time_thres=180)

keep_ids = set(df_stats['TRIP_KEY'])
df_percentile_map = df_percentile_map[df_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops = df_stops[df_stops['TRIP_KEY'].isin(keep_ids)].copy()

df_percentile_map = functions.filter_motion_data_speed_percentile(df_percentile_map, threshold, 'SPEED_P85')

keep_ids_p = set(df_percentile_map['TRIP_KEY'])
df_stats = df_stats[df_stats['TRIP_KEY'].isin(keep_ids_p)].copy()
df_stops = df_stops[df_stops['TRIP_KEY'].isin(keep_ids_p)].copy()

# Intermediatory validation (should be True True):
print(len(df_stats) == len(df_percentile_map) == len(df_stops))
print(sorted(df_stats['TRIP_KEY']) == sorted(df_percentile_map['TRIP_KEY']) == sorted(df_stops['TRIP_KEY']))

# Here we create our feature dataset that we want to evaluate on the mode, we are fusing our 3 engineered datasets-
# the stops, the motion features and the optimal percentiles.
# We end up with "df_merge" which contains all the information we need to train and evaluate the models.
# Only load the optimal cols
best_p, js_agg, all_js = percentile_calculate.calculate_percentile_distributions(df_pm = df_percentile_map, remap = False, mode = 'mean')
percentile_optimal_list = best_p['col'].tolist()
percentile_optimal_list.append('TRIP_KEY')

df_percentile_optimal = df_percentile_map[percentile_optimal_list]

drop_stat_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_AVG', 'B_MED', 'B_85P', 'B_STD', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_AVG', 'BR_MED', 'BR_85P', 'BR_STD', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'BC_AVG', 'BC_MED', 'BC_85P', 'BC_STD', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84', 'AVG_PT_DIST', 'MED_PT_DIST', 'DIST_TOTAL']
df_stats_slim = df_stats.drop(columns=drop_stat_cols)

# Stops
best_p_stop, js_agg_stop, all_js_stop = calculate_optimal_p_stops.calculate_percentile_distributions(df_pm = df_stops, bins=50, mode='mean')
percentile_optimal_list_stop = best_p_stop['col'].tolist()
percentile_optimal_list_stop.append('TRIP_KEY')
df_stops_optimal = df_stops[percentile_optimal_list_stop]

# Find al _Pxx stops
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too 
df_stops_dropped = df_stops.drop(columns=drop_cols_Pxx)
df_stops_optimal = df_stops_optimal.merge(df_stops_dropped, on='TRIP_KEY', how='left', validate='one_to_one')

# Join the perctile_opt map to the df_stats ->
df_merge = df_stats_slim.merge(df_percentile_optimal, on='TRIP_KEY', how='left', validate='one_to_one')
df_merge = df_merge.merge(df_stops_optimal, on='TRIP_KEY', how='left', validate='one_to_one')

# Re introduce non-WGS84 in optimal percentiles - bit inefficient but we drop them again:
merge_drop = ['BC_P50', 'BR_P50']
df_merge = df_merge.drop(columns=merge_drop)


# ------ INTRODUCE ROUTE-MATCHED COLS ----
# Load the correct data for map matching:
df_mapmatched = pd.read_csv("/home/s232534/scripts/map_matching/map_matching_interpolated_percentages.csv") 
df_mapmatched = df_mapmatched.drop(columns=['MAIN_MODE', 'MULTI_MODAL'])

# Merge in our new feature cols:
df_merge_matched = df_merge.merge(df_mapmatched, on='TRIP_KEY', how='left', validate='one_to_one')

# Currently non-matched cols are NaN, fill with zero
cols = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_merge_matched[cols] = df_merge_matched[cols].fillna(0)

At WALKING threshold 6.5; 83 are above.
At BIKE threshold 10; 7 are above.
At CAR threshold 39; 7 are above.
At BUS threshold 14; 13 are above.
At TRAMWAY threshold 19.5; 0 are above.
At SUBWAY threshold 11.1; 8 are above.
At TRAIN threshold 61; 4 are above.
Filtered by TOTAL_TIME > 180. Dropped 4587 rows.
At WALKING threshold 6.5; 361 are above.
At BIKE threshold 10; 46 are above.
At CAR threshold 39; 10 are above.
At BUS threshold 14; 154 are above.
At TRAMWAY threshold 19.5; 5 are above.
At SUBWAY threshold 11.1; 64 are above.
At TRAIN threshold 61; 16 are above.
True
True
          col   mode_a   mode_b        js
0   SPEED_P50   SUBWAY  WALKING  0.004241
1   SPEED_P50   SUBWAY    TRAIN  0.100848
2   SPEED_P50   SUBWAY      BUS  0.041378
3   SPEED_P50   SUBWAY      CAR  0.396111
4   SPEED_P50   SUBWAY     BIKE  0.321445
5   SPEED_P50   SUBWAY  TRAMWAY  0.049499
6   SPEED_P50  WALKING    TRAIN  0.093467
7   SPEED_P50  WALKING      BUS  0.039190
8   SPEED_P50  WALKING      CAR  0.3990

In [ ]:
# Train our model on our latest features dataset for single-modal
xgb_single_modal, y_pred_sm, y_test_sm, class_names_sm = trip_classification.run_xgb_classifier(df_merge_matched)

f1_sm = f1_score(y_test_sm, y_pred_sm, average="macro")
print("Macro F1 simple :", f1_sm)

print("\nClassification report for simple:")
print(classification_report(y_test_sm, y_pred_sm, target_names=class_names_sm))

Macro F1 simple : 0.7798308706087315

Classification report for simple:
              precision    recall  f1-score   support

        BIKE       0.84      0.78      0.81       720
         BUS       0.74      0.58      0.65       489
         CAR       0.94      0.92      0.93      4717
      SUBWAY       0.77      0.58      0.66       733
       TRAIN       0.86      0.78      0.82       462
     TRAMWAY       0.76      0.68      0.72        98
     WALKING       0.82      0.92      0.87      3418

    accuracy                           0.87     10637
   macro avg       0.82      0.75      0.78     10637
weighted avg       0.87      0.87      0.86     10637

Saving model...


In [3]:
f1score, xgb_model_single_mode, le = trip_classification.run_xgb_classifier_kfold(df_merge_matched, 5, return_le=True)

# Save the model:
MOUT = Path("/home/s232534/scripts/segmentation_pelt/multimodal_classification/xgb_singlemodal_classifier.ubj")
print("Saving model...")
xgb_model_single_mode.save_model(MOUT)

# Save out the label encoder so that we can rebuild our human readable labels as well
# Couldn't find a good way so using joblib:
joblib.dump(le, "/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib")

Saving model...


['/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib']

Top single modal model performance = 

In [4]:
print(f1score)

0.7948497621745395


# Segment based classification:

- Simple example based on one trace

In [11]:
# Limit to only multimodal
df_segments = df_segments[df_segments['MULTI_MODAL'] == True]

# Lets just take the second trip in the multi-modal df
trip_key = df_segments['KEY'].iloc[1]
segment_boundaries = df_segments['SEGMENTS_BOUNDS'].iloc[1]

# Load the relevant GPS file to that trip and isolate the trip:
df_gps = pd.read_parquet("/home/s232534/data/split_motion_features_061125/10_2980_s_motion.parquet")
df_trip = df_gps[df_gps['KEY'] == trip_key]
df_trip_info = df_trips_dataset[df_trips_dataset['KEY'] == trip_key]

In [28]:
df_trip_info['Mode_3']

35333    NaN
Name: Mode_3, dtype: object

We need the trip info to get our set of ground truths which is the main_mode + any other listed modes.

In [24]:
ground_truth_label_set = []
ground_truth_label_set.append(df_trip_info['Main_Mode'].values[0])

for i in range(1, 6):
    col = f"Mode_{i}"
    val = df_trip_info[col].values[0]

    if pd.notna(val):
        ground_truth_label_set.append(val)

In [29]:
print(ground_truth_label_set)

['SUBWAY', 'SUBWAY', 'SUBWAY']


In [ ]:
# We only check for uniques since we do not know the order of events.
print(set(ground_truth_label_set))

# also if ground truth set does not include walking we will ignore it 
# from classification because its very likely in multi-modal journies that walking occurs between transport and segmentation would have captures it.

{'SUBWAY'}


# Regenerate Features on Segment Level

## Motion Features

In [ ]:
# First we require the trip segment boundaries, this is using the signal of SPEED and for certain trips 
# the generated 0/1 signal from map matching. 
# Output = trip_segment_boundaries_pelt.csv
%run /home/s232534/scripts/segmentation_pelt/generate_segment_all.py

# Run the motion features table generator this gives us trip-level -> segmeent-level notion features
# Generates using the boundaries defined in: trip_segment_boundaries_pelt.csv. 
# Output = features_motion_segmented.parquet/csv
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_motion_features.py

150
70
60
100
180
30
50
130
41


In [33]:
# The output is now something like this:
df_motion_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_motion_segmented.parquet")
df_motion_features.head(15)

,ID,FILE_NAME,TRIP_KEY,SEGMENT,MAIN_MODE,MULTI_MODAL,TOTAL_TIME,AVG_PT_DIST_WGS84,MED_PT_DIST_WGS84,DIST_TOTAL_WGS84,...,B_85P_WGS84,B_STD_WGS84,BC_AVG_WGS84,BC_MED_WGS84,BC_85P_WGS84,BC_STD_WGS84,BR_AVG_WGS84,BR_MED_WGS84,BR_85P_WGS84,BR_STD_WGS84
0,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,0,WALKING,False,98.0,4.681348,2.637832,154.484469,...,293.337282,25.141284,16.987711,7.812332,31.314090,21.229647,8.493856,3.906166,15.657045,10.614824
1,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,1,WALKING,False,158.0,2.888864,2.756887,231.109127,...,295.812961,117.058179,20.332189,17.198196,38.293977,16.816197,10.166095,8.599098,19.146989,8.408099
2,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,2,WALKING,False,78.0,2.833693,2.762952,113.347733,...,347.939884,157.323120,15.274170,13.518416,25.222374,12.736938,7.637085,6.759208,12.611187,6.368469
3,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,3,WALKING,False,80.0,5.204981,2.701627,145.739480,...,329.311598,66.724008,28.096828,17.461901,49.071335,32.291468,14.048414,8.730950,24.535667,16.145734
4,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,0,WALKING,False,79.0,23.326380,13.758221,676.465022,...,345.408040,147.349790,16.204709,9.545649,24.646429,27.708984,8.060237,4.772824,12.323214,13.870665
5,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,1,WALKING,False,98.0,11.774936,2.861874,153.074164,...,258.403773,58.723772,46.097288,36.093222,90.399453,32.830157,23.048644,18.046611,45.199726,16.415079
6,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,2,WALKING,False,218.0,5.642264,2.425640,541.657366,...,278.198437,112.371931,44.461933,30.283461,105.026915,44.321179,22.230966,15.141730,52.513457,22.160589
7,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,3,WALKING,False,138.0,14.350887,2.947761,71.754435,...,344.391571,178.882014,23.383514,20.330406,36.853256,13.469963,11.691757,10.165203,18.426628,6.734981
8,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,4,WALKING,False,58.0,4.020818,4.492958,120.624533,...,187.069990,112.585880,25.128186,10.575169,29.032965,39.807309,12.564093,5.287584,14.516482,19.903654
9,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,5,WALKING,False,78.0,4.031339,2.267829,161.253577,...,339.668312,124.733302,73.866673,58.953397,153.722744,62.501350,36.933337,29.476699,76.861372,31.250675


This way we can now have segment level motion feature summaries, so we filter by KEY/ID then my the segment.

## Optimal percentiles: motion features

- Upon further thinking computing the percentile map further divided into segments is correct. BUT we do not need to re-calculate optimal percentiles on this new map because we are still to treat them as single-modal essentially, so we can simply re-use the resulting columns from the previous single-modal only computations and just attach the new cols from the percentile map segmented :)

- So TL;DR: we just need the re-computed percentile map that includes segments.

In [3]:
# Once more we now regenerate our percentile map for our features on the segment level
# Output = percentile_map_motion_segmented.parquet
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_percentile_map_segmented.py

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [00:59<00:00, 55.57file/s] 


In [2]:
# The data looks like this:
df_percentile_map_segmented = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/percentile_map_motion_segmented.parquet")
df_percentile_map_segmented.head(15)

,ID,FILE_NAME,TRIP_KEY,SEGMENT,MAIN_MODE,MULTI_MODAL,SPEED_P50,SPEED_P55,SPEED_P60,SPEED_P65,...,BR_WGS84_P50,BR_WGS84_P55,BR_WGS84_P60,BR_WGS84_P65,BR_WGS84_P70,BR_WGS84_P75,BR_WGS84_P80,BR_WGS84_P85,BR_WGS84_P90,BR_WGS84_P95
0,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,0,WALKING,False,0.715556,0.726792,0.751056,0.830153,...,3.906166,5.542074,6.750772,8.926008,9.739394,11.691104,12.240551,15.657045,18.875540,33.020972
1,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,1,WALKING,False,0.931806,0.952708,0.970389,1.014292,...,8.599098,9.470496,10.574945,11.298146,11.973580,13.574597,16.319811,19.146989,20.869664,27.745768
2,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,2,WALKING,False,1.383472,1.411417,1.477778,1.508570,...,6.759208,7.200088,7.830867,8.984125,10.440987,11.690407,12.411899,12.611187,14.694553,15.638270
3,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,3,WALKING,False,1.017222,1.038333,1.047778,1.081111,...,8.730950,8.882903,8.968067,12.356778,17.312582,17.639581,21.472873,24.535667,34.984128,39.104859
4,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,0,WALKING,False,5.184445,5.581903,5.821111,5.991847,...,4.772824,6.023348,6.691444,7.664173,8.112139,8.751977,10.522150,12.323214,12.696670,16.907752
5,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,1,WALKING,False,0.778056,0.806319,0.863167,0.915347,...,18.046611,18.599178,20.240348,22.165151,24.846311,30.897576,40.992966,45.199726,46.041556,48.825080
6,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,2,WALKING,False,1.734444,1.858903,2.042611,2.161792,...,15.141730,17.851993,19.079688,21.351092,24.195281,26.554170,36.800778,52.513457,59.219960,70.576137
7,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,3,WALKING,False,0.815139,0.848722,0.952167,1.047000,...,10.165203,11.771067,13.376930,14.982794,16.588658,18.194522,18.310575,18.426628,18.542681,18.658734
8,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,4,WALKING,False,3.307084,3.533806,3.606667,3.681500,...,5.287584,6.738737,7.977177,10.604069,12.484248,13.065764,13.289006,14.516482,22.917336,53.810706
9,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,5,WALKING,False,1.620972,1.712903,1.904778,2.161250,...,29.476699,33.451962,37.972645,44.266363,58.251640,75.501963,75.901072,76.861372,83.070709,85.907656


In [ ]:
# NOTE: WE DO NOT NEED TO DO THIS JUST USE PREVIOUS COMPUTATIONS AS THE COLS :)
# We can then use the newly generated map to find optimal percentiles 
# for all trips and their segments
best_p, js_agg, all_js = percentile_calculate.calculate_percentile_distributions_segmented(df_pm = df_percentile_map_segmented, remap = False, mode = 'mean')
percentile_optimal_list_segmented = best_p['col'].tolist()
percentile_optimal_list_segmented.append('TRIP_KEY')

          col           mode_a              mode_b        js
0   SPEED_P50          WALKING     PRIV_CAR_DRIVER  0.334445
1   SPEED_P50          WALKING              SUBWAY  0.035709
2   SPEED_P50          WALKING          ELECT_BIKE  0.259274
3   SPEED_P50          WALKING               OTHER  0.238400
4   SPEED_P50          WALKING               TRAIN  0.177326
5   SPEED_P50          WALKING  PRIV_CAR_PASSENGER  0.323649
6   SPEED_P50          WALKING                BIKE  0.244356
7   SPEED_P50          WALKING                 BUS  0.130459
8   SPEED_P50          WALKING             TRAMWAY  0.149550
9   SPEED_P50          WALKING       TRAIN_EXPRESS  0.209671
10  SPEED_P50          WALKING         TWO_WHEELER  0.314597
11  SPEED_P50          WALKING                TAXI  0.266773
12  SPEED_P50          WALKING           ON_DEMAND  0.272177
13  SPEED_P50          WALKING       ELECT_SCOOTER  0.181120
14  SPEED_P50          WALKING               PLANE  0.315903
15  SPEED_P50          W

# Stop proximity features

In [2]:

%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_stop_features.py

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [14:40<00:00,  3.78file/s] 


In [3]:
# and once again the data looks like this:
df_stop_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_stop_proximity.parquet")
df_stop_features.head(15)

,ID,FILE_NAME,TRIP_KEY,SEGMENT,MAIN_MODE,MULTI_MODAL,STOPS_NUM_AVG,STOPS_NUM_MAX,STOPS_NUM_MIN,STOPS_MINDIST_AVG,...,STOPS_CLOSEST_BUS_P50,STOPS_CLOSEST_BUS_P55,STOPS_CLOSEST_BUS_P60,STOPS_CLOSEST_BUS_P65,STOPS_CLOSEST_BUS_P70,STOPS_CLOSEST_BUS_P75,STOPS_CLOSEST_BUS_P80,STOPS_CLOSEST_BUS_P85,STOPS_CLOSEST_BUS_P90,STOPS_CLOSEST_BUS_P95
0,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,0,WALKING,False,38.680000,39.0,38.0,313.345612,...,329.790833,335.480487,339.498621,341.534096,344.942572,348.776718,354.348181,358.630283,359.811716,360.959065
1,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,1,WALKING,False,37.075000,38.0,36.0,284.003845,...,271.741180,278.029311,281.791461,283.392279,287.846881,315.159088,321.983093,326.454419,334.715811,344.837442
2,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,2,WALKING,False,35.675000,37.0,35.0,269.042145,...,271.760925,273.240439,274.207239,274.817924,276.375668,277.057068,277.612537,278.912465,279.465311,279.531232
3,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-1,3,WALKING,False,33.121951,35.0,30.0,198.153885,...,199.649078,203.303360,209.055161,215.922882,222.284409,228.338425,230.771515,233.843674,238.327957,241.075378
4,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,0,WALKING,False,29.325000,30.0,27.0,151.434570,...,132.285263,140.159964,144.342453,149.370901,163.838029,176.820068,197.360587,222.999012,237.465062,258.488126
5,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,1,WALKING,False,30.280000,36.0,29.0,117.373070,...,103.233597,103.233597,103.233597,103.233597,103.233597,103.233597,159.158041,162.218423,162.959041,163.614258
6,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,2,WALKING,False,29.854545,30.0,29.0,114.772530,...,108.561703,109.380400,110.080760,111.471469,112.860655,115.285532,123.402536,133.004461,146.779538,152.941599
7,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,3,WALKING,False,29.071429,30.0,29.0,104.559692,...,103.233597,103.233597,103.233597,103.233597,103.233597,103.233597,103.233597,103.233597,103.233597,121.031393
8,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,4,WALKING,False,30.000000,30.0,30.0,154.792236,...,162.108795,164.015402,165.360010,166.832274,167.399913,167.621891,168.395782,170.721727,171.999507,174.925249
9,10_2978,10_2978_s_motion.parquet,10_2978-friday-45002-2,5,WALKING,False,30.000000,30.0,30.0,167.862396,...,179.010834,179.658971,180.396558,180.918490,181.464142,182.830719,184.308835,184.819671,185.555312,186.288857


# Route-matching with segments

In [4]:
# Finally, we run our last feature generation, the route-matching
# Output = features_route_matching.parquet
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_route_match_features.py

Loading route parquet files...
Loading segment boundaries...
Multi-threading: Interpolating routes...
Generating BallTree for bus network...
Generating BallTree for tram network...
Generating BallTree for train network...
Generating BallTree for subway network...


Processing: 100%|██████████| 3324/3324 [02:23<00:00, 23.19file/s]


In [21]:
# Quickly investigate the data:
df_route_match_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_route_matching.parquet")

# Ensure the NaNs are filled with 0
cols = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_route_match_features[cols] = df_route_match_features[cols].fillna(0)

df_route_match_features.head(15)

,KEY,SEGMENT,bus_match_pct,tram_match_pct,train_match_pct,subway_match_pct,MAIN_MODE,MULTI_MODAL
0,10_2978-friday-45002-1,0,0.000,0.0,0.0,0.0,WALKING,False
1,10_2978-friday-45002-1,1,0.000,0.0,0.0,0.0,WALKING,False
2,10_2978-friday-45002-1,2,0.000,0.0,0.0,0.0,WALKING,False
3,10_2978-friday-45002-1,3,0.341,0.0,0.0,0.0,WALKING,False
4,10_2978-friday-45002-2,0,0.175,0.0,0.0,0.0,WALKING,False
5,10_2978-friday-45002-2,1,0.000,0.0,0.0,0.0,WALKING,False
6,10_2978-friday-45002-2,2,0.636,0.0,0.0,0.0,WALKING,False
7,10_2978-friday-45002-2,3,0.057,0.0,0.0,0.0,WALKING,False
8,10_2978-friday-45002-2,4,0.367,0.0,0.0,0.0,WALKING,False
9,10_2978-friday-45002-2,5,0.000,0.0,0.0,0.0,WALKING,False


# Multi-modal classification

Now we have all the pieces of the puzzle to perform classification of transport in GPS traces that contain multiple different modes, and transportation switches. We do this again using the segments and treating them as single modes, so each segment provides its split motion features and we apply our pre-trained single-modal XGBoost model to get a classifcation label.

## Building the features dataframe

In [6]:
# First we must build up a complete segment feature table 
# as we have previously done with the non-segmented trips

# For posterity we reload the data:
df_motion_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_motion_segmented.parquet") 
df_motion_percentile_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/percentile_map_motion_segmented.parquet") # _map means it still contains many percentile features that needs to be "optimised"
df_stops_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_stop_proximity.parquet")
df_route_match_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/features_route_matching.parquet")

# filter for multi-modal .
df_motion_features = df_motion_features.loc[df_motion_features['MULTI_MODAL']] 
df_motion_percentile_map = df_motion_percentile_map.loc[df_motion_percentile_map['MULTI_MODAL']] 
df_stops_map = df_stops_map.loc[df_stops_map['MULTI_MODAL']] 
df_route_match_features = df_route_match_features.loc[df_route_match_features['MULTI_MODAL']] 

# Drop modes we will not be using for this thesis and modes to merge and remap:
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_motion_features = functions.remap_and_drop_modes_df(df_motion_features, mode_remap, modes_to_drop)
df_motion_percentile_map = functions.remap_and_drop_modes_df(df_motion_percentile_map, mode_remap, modes_to_drop)
df_stops_map = functions.remap_and_drop_modes_df(df_stops_map, mode_remap, modes_to_drop)
df_route_match_features = functions.remap_and_drop_modes_df(df_route_match_features, mode_remap, modes_to_drop)

# Filter journeys that are short... they will not provide enough data to accurately classify them.
# We can use our non-segmented feature table to extract the total times.
df_motion_features_nonsegmented = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_motion_features_nonsegmented = df_motion_features_nonsegmented.loc[df_motion_features_nonsegmented['MULTI_MODAL']] 

#df_motion_features_nonsegmented = functions.filter_motion_data_speed_avg(df_motion_features_nonsegmented, threshold) We can't really filter by mean because we don't know what modes are exactly present
df_motion_features_nonsegmented = functions.filter_temporal_data(df_motion_features_nonsegmented, time_thres=180)

# We can propogate this filtering by taking the remaining IDs as a mask essentially
keep_ids = set(df_motion_features_nonsegmented['TRIP_KEY'])

df_motion_features = df_motion_features[df_motion_features['TRIP_KEY'].isin(keep_ids)].copy()
df_motion_percentile_map = df_motion_percentile_map[df_motion_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_map = df_stops_map[df_stops_map['TRIP_KEY'].isin(keep_ids)].copy()
df_route_match_features = df_route_match_features[df_route_match_features['KEY'].isin(keep_ids)].copy()

# Intermediatory validation (should be True True):
t1: bool = (len(df_motion_features) == len(df_motion_percentile_map) == len(df_stops_map) == len(df_route_match_features))
t2: bool = (sorted(df_motion_features['TRIP_KEY']) == sorted(df_motion_percentile_map['TRIP_KEY']) == sorted(df_stops_map['TRIP_KEY']) == sorted(df_route_match_features['KEY']))
assert t1 and t2, f"Validation FAILED: length={t1}, keys={t2}"

# Finally we need to extract our optimal percentiles from the maps. Since we are classifying under the guise of single-modal journeys
# we can just re-use our single-modal optimal percentiles we calculated for training the single-modal classifier.
# this is contained in the 'percentile_optimal_list' and 'percentile_optimal_list_stop' found in the earlier cells.
# NOTE: We need to remove the features cols we do not use and are not generated upstream in these new scripts. (BC_ and BR_ non-WGS84)

percentile_optimal_list_segment = percentile_optimal_list.copy() # safer to copy
percentile_optimal_list_stop_segment = percentile_optimal_list_stop.copy()

percentile_optimal_list_segment.append('SEGMENT')
percentile_optimal_list_stop_segment.append('SEGMENT') # <- Need to add the segment col too so that we don't lose it as we are no linking on ID and SEGMENT

drop = {"BC_P50", "BR_P50"}
percentile_optimal_list_segment = [c for c in percentile_optimal_list_segment if c not in drop]

df_optimal_motion_features = df_motion_percentile_map[percentile_optimal_list_segment]
df_optimal_stop_features = df_stops_map[percentile_optimal_list_stop_segment]

# Rebuild the stops
# Find all _Pxx stops we remove then -> re-introduce the optimal ones again.
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_map.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too otherwise we double up.
df_stops_dropped = df_stops_map.drop(columns=drop_cols_Pxx)
df_stops_features = df_optimal_stop_features.merge(df_stops_dropped, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Rebuild the full motion features including the optimal percentiles:
drop_motion_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84']
df_motion_features = df_motion_features.drop(columns=drop_motion_cols)
df_motion_features = df_motion_features.merge(df_optimal_motion_features, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Fix up route matching dataset to be ready to merge in
df_route_match_features.rename(columns={"KEY": "TRIP_KEY"}, inplace=True) # Should be fixed upstream but the name does not align with the standard.
df_route_match_features = df_route_match_features.drop(columns=['MAIN_MODE', 'MAIN_MODE_ORIG', 'MULTI_MODAL'])
cols_rm = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_route_match_features[cols_rm] = df_route_match_features[cols_rm].fillna(0)

# Build up the final dataframe ready for modelling:
df_full_segmented_features = (
    df_motion_features
      .merge(df_stops_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
      .merge(df_route_match_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
)

Filtered by TOTAL_TIME > 180. Dropped 120 rows.


We now have our full dataset

In [7]:
df_full_segmented_features.keys()

Index(['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'TOTAL_TIME', 'AVG_PT_DIST_WGS84',
       'MED_PT_DIST_WGS84', 'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG',
       'SPEED_MED', 'SPEED_STD', 'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG',
       'JERK_MED', 'JERK_STD', 'BC_AVG_WGS84', 'BC_MED_WGS84', 'BC_STD_WGS84',
       'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84', 'ACC_P95',
       'BC_WGS84_P50', 'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95',
       'STOPS_CLOSEST_BUS_P95', 'STOPS_CLOSEST_METRO_P95',
       'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95',
       'STOPS_NUM_P5', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN',
       'STOPS_MINDIST_AVG', 'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL',
       'DURATION_BUS', 'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL',
       'MIN_DIST_BUS', 'bus_match_pct', 'tram_match_pct', 'train_match_pct',
       'subway_match_pct'],
      dtype='object')

## Classifying multi-modal on segment level

In [8]:
# Just for clarity we reload the classifer that was saved earlier in the book
xgb_single_classifier = XGBClassifier()
xgb_single_classifier.load_model("/home/s232534/scripts/segmentation_pelt/multimodal_classification/xgb_singlemodal_classifier.ubj")
le = joblib.load("/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib")

# Our feature set
X = df_full_segmented_features.copy()
X = X.drop(columns=['TRIP_KEY', 'SEGMENT', 'MAIN_MODE'])

# Run prediction:
proba = xgb_single_classifier.predict_proba(X) 
idx = proba.argmax(axis=1)

# Fill df with results:
df_full_segmented_results = df_full_segmented_features[['TRIP_KEY', 'SEGMENT', 'MAIN_MODE']].copy()
 
df_full_segmented_results["PRED_MODE"] = le.inverse_transform(idx)
df_full_segmented_results["PRED_CONF"] = proba.max(axis=1)

In [ ]:
# Batch the results into sets

df_full_segmented_results_sets = (
    df_full_segmented_results.groupby("TRIP_KEY")["PRED_MODE"]
    .agg(lambda s: set(s.dropna()))
    .reset_index(name="PRED_SET")
)

# Create ground truth sets:
df_trips_dataset = pd.read_csv("/home/lauale/netmob2025/data/trips_dataset.csv")
bad_key = "2_1946-monday-44935-1" # Duplicate key that actually stems from the original dataset we need to drop it.
df_trips_dataset = df_trips_dataset.loc[df_trips_dataset["KEY"] != bad_key].copy()

# Before generating the sets we need to remap some names:
# We also for now keep ELECT_BIKE and ELECT_SCOOTER and remap to BIKE
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN',
    'ELECT_BIKE': 'BIKE',
    'ELECT_SCOOTER': 'BIKE',
}
mode_cols = ["Main_Mode", "Mode_1", "Mode_2", "Mode_3", "Mode_4", "Mode_5"]
df_trips_dataset[mode_cols] = df_trips_dataset[mode_cols].replace(mode_remap)

# Build the ground truths
def build_gt_set(row):
    return {row[c] for c in mode_cols if pd.notna(row.get(c))}

df_trips_dataset["GT_SET"] = df_trips_dataset.apply(build_gt_set, axis=1)
# Add ability to filter if we have out of scope labels
labels = {"PLANE", "OTHER"}
df_trips_dataset["OOS"] = df_trips_dataset["GT_SET"].apply(lambda s: isinstance(s, set) and bool(s & labels))

# Merge the two together to have Predicted and GroundTruth sets.
filter_ids = df_full_segmented_results_sets['TRIP_KEY'].unique()
df_trips_dataset_filtered = df_trips_dataset.loc[df_trips_dataset["KEY"].isin(filter_ids), ["KEY", "GT_SET", "OOS"]]
df_trips_dataset_filtered.rename(columns={"KEY": "TRIP_KEY"}, inplace=True)

df_full_segmented_results_sets = df_full_segmented_results_sets.merge(df_trips_dataset_filtered, on=["TRIP_KEY"], how="left", validate="one_to_one")

In [17]:
# Jacquard scoring test:
def jaccard(pred, gt):
    if not isinstance(pred, set) or not isinstance(gt, set):
        return np.nan
    if len(pred | gt) == 0:
        return np.nan
    return len(pred & gt) / len(pred | gt)

def jaccard_exclude_extra_walking(pred: set, gt: set) -> float:
    if not isinstance(pred, set) or not isinstance(gt, set):
        return np.nan
    pred2 = set(pred)
    gt2 = set(gt)

    # Exclude walking only if GT doesn't contain it
    if 'WALKING' not in gt2:
        pred2.discard('WALKING')

    union = pred2 | gt2
    if not union:
        return np.nan
    return len(pred2 & gt2) / len(union)

In [ ]:
df_full_segmented_results_sets["JACCARD"] = df_full_segmented_results_sets.apply(
    lambda r: jaccard_exclude_extra_walking(r["PRED_SET"], r["GT_SET"]), axis=1
)

In [69]:
print("Full DS Jaccard mean:")
print(df_full_segmented_results_sets['JACCARD'].mean())

print("Without labels like Plane and Other DS Jaccard mean:")

print(df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]['JACCARD'].mean())

Full DS Jaccard mean:
0.4160934458144062
Without labels like Plane and Other DS Jaccard mean:
0.4165038850544776


In [ ]:
# Calculate percentage of exact matches
exact_match_pct = len(df_full_segmented_results_sets[df_full_segmented_results_sets['JACCARD'] == 1]) / len(df_full_segmented_results_sets['JACCARD'])
print(f"Exact match (%): {exact_match_pct*100}")

Exact match (%): 7.7601868673760706


In [71]:
df_full_segmented_results_sets.to_csv("/home/s232534/scripts/segmentation_pelt/multimodal_classification/multi-modal-classification-RESULTS.csv")

# Re-run with varying segmentation strategies:

### Segmentation 

In [1]:
# Speed only segmentation, output = trip_segment_boundaries_pelt_SPEEDONLY.csv
%run /home/s232534/scripts/segmentation_pelt/generate_segment_all_SPEED.py

Processing: 100%|██████████| 3324/3324 [1:16:30<00:00,  1.38s/file]  


In [ ]:
# Speed and bearing rate segmentation, output = trip_segment_boundaries_pelt_SPEED-BR.csv
%run /home/s232534/scripts/segmentation_pelt/generate_segment_all_SPEED-BR.py

Processing:   7%|▋         | 236/3324 [5:22:28<94:38:35, 110.34s/file]

In [ ]:
# Speed and bearing rate segmentation and route-match signals, output = trip_segment_boundaries_pelt_SIGNAL.csv
%run /home/s232534/scripts/segmentation_pelt/generate_segment_all_SIGNAL_nobranch.py

### Motion features:

In [ ]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_motion_features.py \
  --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEEDONLY.csv \
  --out-stem features_motion_segmented_SPEEDONLY.parquet

Processing: 100%|██████████| 3324/3324 [01:33<00:00, 35.74file/s]


In [2]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_motion_features.py \
  --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEED-BR.csv \
  --out-stem features_motion_segmented_SPEED-BR.parquet

Processing: 100%|██████████| 3324/3324 [02:13<00:00, 24.95file/s]


In [1]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_motion_features.py \
  --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SIGNAL-030126.csv \
  --out-stem features_motion_segmented_SIGNAL.parquet

Processing: 100%|██████████| 3324/3324 [02:01<00:00, 27.35file/s]


Check the mean amount of segments 

In [ ]:
testtt = pd.read_csv("/home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEEDONLY.csv")
s = testtt["SEGMENTS_BOUNDS"].apply(ast.literal_eval)   
mean_n_breakpoints = s.apply(len).mean()
print("Mean segments for speed-only")
print(mean_n_breakpoints)

Mean segments for speed-only
3.7288331336345655


In [ ]:
testtt = pd.read_csv("/home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEED-BR.csv")
s = testtt["SEGMENTS_BOUNDS"].apply(ast.literal_eval)  
mean_n_breakpoints = s.apply(len).mean()
print("Mean segments for speed and bearing change rate")
print(mean_n_breakpoints)

Mean segments for speed and bearing change rate
3.2449365535361117


In [3]:
testtt = pd.read_csv("/home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SIGNAL-030126.csv")
s = testtt["SEGMENTS_BOUNDS"].apply(ast.literal_eval)   
mean_n_breakpoints = s.apply(len).mean()
print("Mean segments for speed and bearing change rate")
print(mean_n_breakpoints)

Mean segments for speed and bearing change rate
3.5258477573587794


Size of motion feature tables:

In [4]:
mf_speedonly = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEEDONLY.parquet")
mf_speedbr = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEED-BR.parquet")
mf_rsignal = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SIGNAL.parquet")

print(f"Speed only len: {len(mf_speedonly)}  |  Speed-BR len: {len(mf_speedbr)}  |  Routes: {len(mf_rsignal)}")

Speed only len: 258300  |  Speed-BR len: 224780  |  Routes: 244239


### Optimal percentiles

In [8]:
# Generate the percentile maps for our 3 segmentation rulesets
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_percentile_map_segmented.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEEDONLY.csv \
    --out-stem percentile_map_segmented_SPEEDONLY.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [00:38<00:00, 85.49file/s] 


In [9]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_percentile_map_segmented.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEED-BR.csv \
    --out-stem percentile_map_segmented_SPEED-BR.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [00:35<00:00, 92.47file/s] 


In [5]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_percentile_map_segmented.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SIGNAL-030126.csv \
    --out-stem percentile_map_segmented_SIGNAL.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [00:35<00:00, 94.13file/s] 


Validation

In [6]:
so = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEEDONLY.parquet")
sbr = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEED-BR.parquet")
rs = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SIGNAL.parquet")

print(f"Speed only len: {len(so)}  |  Speed-BR len: {len(sbr)}  |  Route signal len: {len(rs)}")

Speed only len: 258300  |  Speed-BR len: 224780  |  Route signal len: 244239


### Stop proximity features

In [16]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_stop_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEEDONLY.csv \
    --out-stem features_stop_segmented_SPEEDONLY.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [11:28<00:00,  4.83file/s] 


In [17]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_stop_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEED-BR.csv \
    --out-stem features_stop_segmented_SPEED-BR.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [10:59<00:00,  5.04file/s] 


In [7]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_stop_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SIGNAL-030126.csv \
    --out-stem features_stop_segmented_SIGNAL.parquet

Using:  56  CPUs...


Processing: 100%|██████████| 3324/3324 [10:46<00:00,  5.14file/s] 


In [8]:
so = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SPEEDONLY.parquet")
sbr = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SPEED-BR.parquet")
rs = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SIGNAL.parquet")

print(f"Speed only len: {len(so)}  |  Speed-BR len: {len(sbr)}  |  Route signal len: {len(rs)}")

Speed only len: 258300  |  Speed-BR len: 224780  |  Route signal len: 244239


### Route-matching features

In [19]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_route_match_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEEDONLY.csv \
    --out-stem features_route_match_segmented_SPEEDONLY.parquet

Loading route parquet files...
Loading segment boundaries...
Multi-threading: Interpolating routes...
Generating BallTree for bus network...
Generating BallTree for tram network...
Generating BallTree for train network...
Generating BallTree for subway network...


Processing: 100%|██████████| 3324/3324 [03:21<00:00, 16.51file/s]


In [20]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_route_match_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SPEED-BR.csv \
    --out-stem features_route_match_segmented_SPEED-BR.parquet

Loading route parquet files...
Loading segment boundaries...
Multi-threading: Interpolating routes...
Generating BallTree for bus network...
Generating BallTree for tram network...
Generating BallTree for train network...
Generating BallTree for subway network...


Processing: 100%|██████████| 3324/3324 [03:07<00:00, 17.75file/s]


In [9]:
%run /home/s232534/scripts/segmentation_pelt/multimodal_classification/generate_route_match_features.py \
    --segments /home/s232534/scripts/segmentation_pelt/trip_segment_boundaries_pelt_SIGNAL-030126.csv \
    --out-stem features_route_match_segmented_SIGNAL.parquet

Loading route parquet files...
Loading segment boundaries...
Multi-threading: Interpolating routes...
Generating BallTree for bus network...
Generating BallTree for tram network...
Generating BallTree for train network...
Generating BallTree for subway network...


Processing: 100%|██████████| 3324/3324 [02:05<00:00, 26.48file/s]


In [10]:
so = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SPEEDONLY.parquet")
sbr = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SPEED-BR.parquet")
sr = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SIGNAL.parquet")
print(f"Speed only len: {len(so)}  |  Speed-BR len: {len(sbr)}  |  Route signal len: {len(sr)}")

Speed only len: 258300  |  Speed-BR len: 224780  |  Route signal len: 244239


## Rebuild the full feature dataframe for each segment set and evaluate

Validation / accuracy framework

In [11]:
def compute_main_mode_accuracies(df_full_segmented_results: pd.DataFrame):
    """
    Compute two trip-level dominant-mode accuracy scores:
      1) Confidence-only dominance: sum PRED_CONF per (TRIP_KEY, PRED_MODE)
      2) GPS-points x confidence dominance: sum (GPS_POINTS * PRED_CONF) per (TRIP_KEY, PRED_MODE)
        This is a weighted score essentially that will value labels with higher confidence and GPS point count (larger segment)
        based on the assumption that the main_mode label in the Netmob25 dataset is the longest transport mode taken in a trip.

    Returns
    -------
    results : dict
        {
          "acc_conf_only": float,
          "acc_points_x_conf": float,
          "eval_conf_only": pd.DataFrame,     # optional: per-trip dominant predictions dataset
          "eval_points_x_conf": pd.DataFrame  # optional: per-trip dominant predictions + score dataset
        }
    """
    required = {"TRIP_KEY", "MAIN_MODE", "PRED_MODE", "PRED_CONF"}
    missing = required - set(df_full_segmented_results.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # --- Confidence only dominance ----
    dominant_conf = (
        df_full_segmented_results
        .groupby(["TRIP_KEY", "PRED_MODE"], as_index=False)["PRED_CONF"]
        .sum()
        .sort_values(["TRIP_KEY", "PRED_CONF"], ascending=[True, False])
        .drop_duplicates("TRIP_KEY")
        .rename(columns={"PRED_MODE": "DOM_PRED_MODE", "PRED_CONF": "DOM_SCORE"})
    )

    eval_conf = (
        df_full_segmented_results[["TRIP_KEY", "MAIN_MODE"]]
        .drop_duplicates()
        .merge(dominant_conf[["TRIP_KEY", "DOM_PRED_MODE", "DOM_SCORE"]], on="TRIP_KEY", how="inner")
    )

    acc_conf_only = (eval_conf["DOM_PRED_MODE"] == eval_conf["MAIN_MODE"]).mean()

    # ----- GPS points count x confidence dominance ---
    if "GPS_POINTS" not in df_full_segmented_results.columns:
        raise ValueError("Missing column 'GPS_POINTS' needed for points×confidence dominance.")

    df = df_full_segmented_results.copy()
    df["SEG_SCORE"] = df["GPS_POINTS"] * df["PRED_CONF"]

    dominant_pts = (
        df.groupby(["TRIP_KEY", "PRED_MODE"], as_index=False)["SEG_SCORE"]
        .sum()
        .sort_values(["TRIP_KEY", "SEG_SCORE"], ascending=[True, False])
        .drop_duplicates("TRIP_KEY")
        .rename(columns={"PRED_MODE": "DOM_PRED_MODE", "SEG_SCORE": "DOM_SCORE"})
    )

    eval_pts = (
        df[["TRIP_KEY", "MAIN_MODE"]]
        .drop_duplicates()
        .merge(dominant_pts[["TRIP_KEY", "DOM_PRED_MODE", "DOM_SCORE"]], on="TRIP_KEY", how="inner")
    )

    acc_points_x_conf = (eval_pts["DOM_PRED_MODE"] == eval_pts["MAIN_MODE"]).mean()

    return {
        "acc_conf_only": float(acc_conf_only),
        "acc_points_x_conf": float(acc_points_x_conf),
        "eval_conf_only": eval_conf,
        "eval_points_x_conf": eval_pts,
    }


### SPEED-ONLY segmentation model

In [21]:
# First we must build up a complete segment feature table 
# as we have previously done with the non-segmented trips

# For posterity we reload the data:
df_motion_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEEDONLY.parquet") 
df_motion_percentile_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/percentile_map_segmented_SPEEDONLY.parquet") # _map means it still contains many percentile features that needs to be "optimised"
df_stops_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SPEEDONLY.parquet")
df_route_match_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SPEEDONLY.parquet")

# filter for multi-modal
df_motion_features = df_motion_features.loc[df_motion_features['MULTI_MODAL']] 
df_motion_percentile_map = df_motion_percentile_map.loc[df_motion_percentile_map['MULTI_MODAL']] 
df_stops_map = df_stops_map.loc[df_stops_map['MULTI_MODAL']] 
df_route_match_features = df_route_match_features.loc[df_route_match_features['MULTI_MODAL']] 

# Drop modes we will not be using for this thesis and modes to merge and remap:
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_motion_features = functions.remap_and_drop_modes_df(df_motion_features, mode_remap, modes_to_drop)
df_motion_percentile_map = functions.remap_and_drop_modes_df(df_motion_percentile_map, mode_remap, modes_to_drop)
df_stops_map = functions.remap_and_drop_modes_df(df_stops_map, mode_remap, modes_to_drop)
df_route_match_features = functions.remap_and_drop_modes_df(df_route_match_features, mode_remap, modes_to_drop)

# Filter journeys that are short... they will not provide enough data to accurately classify them.
# We can use our non-segmented feature table to extract the total times.
df_motion_features_nonsegmented = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_motion_features_nonsegmented = df_motion_features_nonsegmented.loc[df_motion_features_nonsegmented['MULTI_MODAL']] 

#df_motion_features_nonsegmented = functions.filter_motion_data_speed_avg(df_motion_features_nonsegmented, threshold) We can't really filter by mean because we don't know what modes are exactly present
df_motion_features_nonsegmented = functions.filter_temporal_data(df_motion_features_nonsegmented, time_thres=180)

# We can propogate this filtering by taking the remaining IDs as a mask essentially
keep_ids = set(df_motion_features_nonsegmented['TRIP_KEY'])

df_motion_features = df_motion_features[df_motion_features['TRIP_KEY'].isin(keep_ids)].copy()
df_motion_percentile_map = df_motion_percentile_map[df_motion_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_map = df_stops_map[df_stops_map['TRIP_KEY'].isin(keep_ids)].copy()
df_route_match_features = df_route_match_features[df_route_match_features['KEY'].isin(keep_ids)].copy()

# Intermediatory validation (should be True True):
t1: bool = (len(df_motion_features) == len(df_motion_percentile_map) == len(df_stops_map) == len(df_route_match_features))
t2: bool = (sorted(df_motion_features['TRIP_KEY']) == sorted(df_motion_percentile_map['TRIP_KEY']) == sorted(df_stops_map['TRIP_KEY']) == sorted(df_route_match_features['KEY']))
assert t1 and t2, f"Validation FAILED: length={t1}, keys={t2}"

# Finally we need to extract our optimal percentiles from the maps. Since we are classifying under the guise of single-modal journeys
# we can just re-use our single-modal optimal percentiles we calculated for training the single-modal classifier.
# this is contained in the 'percentile_optimal_list' and 'percentile_optimal_list_stop' found in the earlier cells.
# NOTE: We need to remove the features cols we do not use and are not generated upstream in these new scripts. (BC_ and BR_ non-WGS84)
# NOTE: IF re-running only this segment - remember to re-run the single-mode classifier data loading and filtering at a minimum.

percentile_optimal_list_segment = percentile_optimal_list.copy() # safer to copy
percentile_optimal_list_stop_segment = percentile_optimal_list_stop.copy()

percentile_optimal_list_segment.append('SEGMENT')
percentile_optimal_list_stop_segment.append('SEGMENT') # <- Need to add the segment col too so that we don't lose it as we are no linking on ID and SEGMENT

drop = {"BC_P50", "BR_P50"}
percentile_optimal_list_segment = [c for c in percentile_optimal_list_segment if c not in drop]

df_optimal_motion_features = df_motion_percentile_map[percentile_optimal_list_segment]
df_optimal_stop_features = df_stops_map[percentile_optimal_list_stop_segment]

# Rebuild the stops
# Find all _Pxx stops we remove then -> re-introduce the optimal ones again.
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_map.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too otherwise we double up.
df_stops_dropped = df_stops_map.drop(columns=drop_cols_Pxx)
df_stops_features = df_optimal_stop_features.merge(df_stops_dropped, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Rebuild the full motion features including the optimal percentiles:
drop_motion_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84']
df_motion_features = df_motion_features.drop(columns=drop_motion_cols)
df_motion_features = df_motion_features.merge(df_optimal_motion_features, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Fix up route matching dataset to be ready to merge in
df_route_match_features.rename(columns={"KEY": "TRIP_KEY"}, inplace=True) # Should be fixed upstream but the name does not align with the standard.
df_route_match_features = df_route_match_features.drop(columns=['MAIN_MODE', 'MAIN_MODE_ORIG', 'MULTI_MODAL'])
cols_rm = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_route_match_features[cols_rm] = df_route_match_features[cols_rm].fillna(0)

# Build up the final dataframe ready for modelling:
df_full_segmented_features = (
    df_motion_features
      .merge(df_stops_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
      .merge(df_route_match_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
)

Filtered by TOTAL_TIME > 180. Dropped 120 rows.


In [22]:
df_full_segmented_features.keys()

Index(['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'TOTAL_TIME', 'AVG_PT_DIST_WGS84',
       'MED_PT_DIST_WGS84', 'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG',
       'SPEED_MED', 'SPEED_STD', 'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG',
       'JERK_MED', 'JERK_STD', 'BC_AVG_WGS84', 'BC_MED_WGS84', 'BC_STD_WGS84',
       'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84', 'ACC_P95',
       'BC_WGS84_P50', 'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95',
       'STOPS_CLOSEST_BUS_P95', 'STOPS_CLOSEST_METRO_P95',
       'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95',
       'STOPS_NUM_P5', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN',
       'STOPS_MINDIST_AVG', 'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL',
       'DURATION_BUS', 'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL',
       'MIN_DIST_BUS', 'bus_match_pct', 'tram_match_pct', 'train_match_pct',
       'subway_match_pct'],
      dtype='object')

The classifier and the results

In [23]:
# Just for clarity we reload the classifer that was saved earlier in the book
xgb_single_classifier = XGBClassifier()
xgb_single_classifier.load_model("/home/s232534/scripts/segmentation_pelt/multimodal_classification/xgb_singlemodal_classifier.ubj")
le = joblib.load("/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib")

# Our feature set
X = df_full_segmented_features.copy()
X = X.drop(columns=['TRIP_KEY', 'SEGMENT', 'MAIN_MODE'])

# Run prediction:
proba = xgb_single_classifier.predict_proba(X) 
idx = proba.argmax(axis=1)

# Fill df with results:
df_full_segmented_results = df_full_segmented_features[['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'GPS_POINTS']].copy()
 
df_full_segmented_results["PRED_MODE"] = le.inverse_transform(idx)
df_full_segmented_results["PRED_CONF"] = proba.max(axis=1)

# -----------BATCHING AND CREATING GROUND TRUTHS ---------

# Batch the results into sets
# MIN_CONF = 0.5  # choose threshold
# df_full_segmented_results = df_full_segmented_results[
#     df_full_segmented_results["PRED_CONF"] >= MIN_CONF
# ].copy()

df_full_segmented_results_sets = (
    df_full_segmented_results.groupby("TRIP_KEY")["PRED_MODE"]
    .agg(lambda s: set(s.dropna()))
    .reset_index(name="PRED_SET")
)

# Create ground truth sets:
df_trips_dataset = pd.read_csv("/home/lauale/netmob2025/data/trips_dataset.csv")
bad_key = "2_1946-monday-44935-1" # Duplicate key that actually stems from the original dataset we need to drop it.
df_trips_dataset = df_trips_dataset.loc[df_trips_dataset["KEY"] != bad_key].copy()

# Before generating the sets we need to remap some names:
# We also for now keep ELECT_BIKE and ELECT_SCOOTER and remap to BIKE
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN',
    # 'ELECT_BIKE': 'BIKE',
    # 'ELECT_SCOOTER': 'BIKE',
}
mode_cols = ["Main_Mode", "Mode_1", "Mode_2", "Mode_3", "Mode_4", "Mode_5"]
df_trips_dataset[mode_cols] = df_trips_dataset[mode_cols].replace(mode_remap)

# Build the ground truths
def build_gt_set(row):
    return {row[c] for c in mode_cols if pd.notna(row.get(c))}

df_trips_dataset["GT_SET"] = df_trips_dataset.apply(build_gt_set, axis=1)
# Add ability to filter if we have out of scope labels
labels = {"PLANE", "OTHER", "ELECT_BIKE", "ELECT_SCOOTER"}
df_trips_dataset["OOS"] = df_trips_dataset["GT_SET"].apply(lambda s: isinstance(s, set) and bool(s & labels))

# Merge the two together to have Predicted and GroundTruth sets.
filter_ids = df_full_segmented_results_sets['TRIP_KEY'].unique()
df_trips_dataset_filtered = df_trips_dataset.loc[df_trips_dataset["KEY"].isin(filter_ids), ["KEY", "GT_SET", "OOS"]]
df_trips_dataset_filtered.rename(columns={"KEY": "TRIP_KEY"}, inplace=True)

df_full_segmented_results_sets = df_full_segmented_results_sets.merge(df_trips_dataset_filtered, on=["TRIP_KEY"], how="left", validate="one_to_one")

# -----------------JACUARD CALCULATIONS AND APPLY TO DS -----------------------

df_full_segmented_results_sets["JACCARD"] = df_full_segmented_results_sets.apply(lambda r: jaccard_exclude_extra_walking(r["PRED_SET"], r["GT_SET"]), axis=1)

In [24]:
# Summary stats
print("Full DS Jaccard mean:")
print(df_full_segmented_results_sets['JACCARD'].mean())

print("Without labels like Plane and Other DS Jaccard mean:")
print(df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]['JACCARD'].mean())

# Calculate percentage of exact matches
exact_match_pct = len(df_full_segmented_results_sets[df_full_segmented_results_sets['JACCARD'] == 1]) / len(df_full_segmented_results_sets['JACCARD'])
print(f"Exact match (%): {exact_match_pct*100}")

print("Exact match removing modes like plane, elect modes...")
filt = df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]
em_pct_filtered = len(filt[filt['JACCARD'] == 1]) / len(filt['JACCARD'])
print(f"Exact match (%): {em_pct_filtered*100}")

# Save out to file.
df_full_segmented_results_sets.to_csv("/home/s232534/scripts/segmentation_pelt/multimodal_classification/multi-modal-classification-RESULTS-SPEEDONLY.csv")

Full DS Jaccard mean:
0.4654856154012546
Without labels like Plane and Other DS Jaccard mean:
0.4714854347342673
Exact match (%): 16.311964702828966
Exact match removing modes like plane, elect modes...
Exact match (%): 16.77118078719146


In [25]:
results = compute_main_mode_accuracies(df_full_segmented_results)
print("MainModeAccuracy (conf only):", results["acc_conf_only"])
print("MainModeAccuracy (points * conf):", results["acc_points_x_conf"])

MainModeAccuracy (conf only): 0.19906566311964702
MainModeAccuracy (points * conf): 0.2553854139631456


### SPEED-Bearing Rate Change segmentation model

In [26]:
# First we must build up a complete segment feature table 
# as we have previously done with the non-segmented trips

# For posterity we reload the data:
df_motion_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SPEED-BR.parquet") 
df_motion_percentile_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/percentile_map_segmented_SPEED-BR.parquet") # _map means it still contains many percentile features that needs to be "optimised"
df_stops_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SPEED-BR.parquet")
df_route_match_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SPEED-BR.parquet")

# filter for multi-modal
df_motion_features = df_motion_features.loc[df_motion_features['MULTI_MODAL']] 
df_motion_percentile_map = df_motion_percentile_map.loc[df_motion_percentile_map['MULTI_MODAL']] 
df_stops_map = df_stops_map.loc[df_stops_map['MULTI_MODAL']] 
df_route_match_features = df_route_match_features.loc[df_route_match_features['MULTI_MODAL']] 

# Drop modes we will not be using for this thesis and modes to merge and remap:
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_motion_features = functions.remap_and_drop_modes_df(df_motion_features, mode_remap, modes_to_drop)
df_motion_percentile_map = functions.remap_and_drop_modes_df(df_motion_percentile_map, mode_remap, modes_to_drop)
df_stops_map = functions.remap_and_drop_modes_df(df_stops_map, mode_remap, modes_to_drop)
df_route_match_features = functions.remap_and_drop_modes_df(df_route_match_features, mode_remap, modes_to_drop)

# Filter journeys that are short... they will not provide enough data to accurately classify them.
# We can use our non-segmented feature table to extract the total times.
df_motion_features_nonsegmented = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_motion_features_nonsegmented = df_motion_features_nonsegmented.loc[df_motion_features_nonsegmented['MULTI_MODAL']] 

#df_motion_features_nonsegmented = functions.filter_motion_data_speed_avg(df_motion_features_nonsegmented, threshold) We can't really filter by mean because we don't know what modes are exactly present
df_motion_features_nonsegmented = functions.filter_temporal_data(df_motion_features_nonsegmented, time_thres=180)

# We can propogate this filtering by taking the remaining IDs as a mask essentially
keep_ids = set(df_motion_features_nonsegmented['TRIP_KEY'])

df_motion_features = df_motion_features[df_motion_features['TRIP_KEY'].isin(keep_ids)].copy()
df_motion_percentile_map = df_motion_percentile_map[df_motion_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_map = df_stops_map[df_stops_map['TRIP_KEY'].isin(keep_ids)].copy()
df_route_match_features = df_route_match_features[df_route_match_features['KEY'].isin(keep_ids)].copy()

# Intermediatory validation (should be True True):
t1: bool = (len(df_motion_features) == len(df_motion_percentile_map) == len(df_stops_map) == len(df_route_match_features))
t2: bool = (sorted(df_motion_features['TRIP_KEY']) == sorted(df_motion_percentile_map['TRIP_KEY']) == sorted(df_stops_map['TRIP_KEY']) == sorted(df_route_match_features['KEY']))
assert t1 and t2, f"Validation FAILED: length={t1}, keys={t2}"

# Finally we need to extract our optimal percentiles from the maps. Since we are classifying under the guise of single-modal journeys
# we can just re-use our single-modal optimal percentiles we calculated for training the single-modal classifier.
# this is contained in the 'percentile_optimal_list' and 'percentile_optimal_list_stop' found in the earlier cells.
# NOTE: We need to remove the features cols we do not use and are not generated upstream in these new scripts. (BC_ and BR_ non-WGS84)
# NOTE: IF re-running only this segment - remember to re-run the single-mode classifier data loading and filtering at a minimum.

percentile_optimal_list_segment = percentile_optimal_list.copy() # safer to copy
percentile_optimal_list_stop_segment = percentile_optimal_list_stop.copy()

percentile_optimal_list_segment.append('SEGMENT')
percentile_optimal_list_stop_segment.append('SEGMENT') # <- Need to add the segment col too so that we don't lose it as we are no linking on ID and SEGMENT

drop = {"BC_P50", "BR_P50"}
percentile_optimal_list_segment = [c for c in percentile_optimal_list_segment if c not in drop]

df_optimal_motion_features = df_motion_percentile_map[percentile_optimal_list_segment]
df_optimal_stop_features = df_stops_map[percentile_optimal_list_stop_segment]

# Rebuild the stops
# Find all _Pxx stops we remove then -> re-introduce the optimal ones again.
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_map.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too otherwise we double up.
df_stops_dropped = df_stops_map.drop(columns=drop_cols_Pxx)
df_stops_features = df_optimal_stop_features.merge(df_stops_dropped, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Rebuild the full motion features including the optimal percentiles:
drop_motion_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84']
df_motion_features = df_motion_features.drop(columns=drop_motion_cols)
df_motion_features = df_motion_features.merge(df_optimal_motion_features, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Fix up route matching dataset to be ready to merge in
df_route_match_features.rename(columns={"KEY": "TRIP_KEY"}, inplace=True) # Should be fixed upstream but the name does not align with the standard.
df_route_match_features = df_route_match_features.drop(columns=['MAIN_MODE', 'MAIN_MODE_ORIG', 'MULTI_MODAL'])
cols_rm = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_route_match_features[cols_rm] = df_route_match_features[cols_rm].fillna(0)

# Build up the final dataframe ready for modelling:
df_full_segmented_features = (
    df_motion_features
      .merge(df_stops_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
      .merge(df_route_match_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
)

Filtered by TOTAL_TIME > 180. Dropped 120 rows.


In [27]:
df_full_segmented_features.keys()

Index(['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'TOTAL_TIME', 'AVG_PT_DIST_WGS84',
       'MED_PT_DIST_WGS84', 'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG',
       'SPEED_MED', 'SPEED_STD', 'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG',
       'JERK_MED', 'JERK_STD', 'BC_AVG_WGS84', 'BC_MED_WGS84', 'BC_STD_WGS84',
       'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84', 'ACC_P95',
       'BC_WGS84_P50', 'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95',
       'STOPS_CLOSEST_BUS_P95', 'STOPS_CLOSEST_METRO_P95',
       'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95',
       'STOPS_NUM_P5', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN',
       'STOPS_MINDIST_AVG', 'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL',
       'DURATION_BUS', 'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL',
       'MIN_DIST_BUS', 'bus_match_pct', 'tram_match_pct', 'train_match_pct',
       'subway_match_pct'],
      dtype='object')

In [28]:
# Just for clarity we reload the classifer that was saved earlier in the book
xgb_single_classifier = XGBClassifier()
xgb_single_classifier.load_model("/home/s232534/scripts/segmentation_pelt/multimodal_classification/xgb_singlemodal_classifier.ubj")
le = joblib.load("/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib")

# Our feature set
X = df_full_segmented_features.copy()
X = X.drop(columns=['TRIP_KEY', 'SEGMENT', 'MAIN_MODE'])

# Run prediction:
proba = xgb_single_classifier.predict_proba(X) 
idx = proba.argmax(axis=1)

# Fill df with results:
df_full_segmented_results = df_full_segmented_features[['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'GPS_POINTS']].copy()
 
df_full_segmented_results["PRED_MODE"] = le.inverse_transform(idx)
df_full_segmented_results["PRED_CONF"] = proba.max(axis=1)

# -----------BATCHING AND CREATING GROUND TRUTHS ---------

# Batch the results into sets
df_full_segmented_results_sets = (
    df_full_segmented_results.groupby("TRIP_KEY")["PRED_MODE"]
    .agg(lambda s: set(s.dropna()))
    .reset_index(name="PRED_SET")
)

# Create ground truth sets:
df_trips_dataset = pd.read_csv("/home/lauale/netmob2025/data/trips_dataset.csv")
bad_key = "2_1946-monday-44935-1" # Duplicate key that actually stems from the original dataset we need to drop it.
df_trips_dataset = df_trips_dataset.loc[df_trips_dataset["KEY"] != bad_key].copy()

# Before generating the sets we need to remap some names:
# We also for now keep ELECT_BIKE and ELECT_SCOOTER and remap to BIKE
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN',
    # 'ELECT_BIKE': 'BIKE',
    # 'ELECT_SCOOTER': 'BIKE',
}
mode_cols = ["Main_Mode", "Mode_1", "Mode_2", "Mode_3", "Mode_4", "Mode_5"]
df_trips_dataset[mode_cols] = df_trips_dataset[mode_cols].replace(mode_remap)

# Build the ground truths
def build_gt_set(row):
    return {row[c] for c in mode_cols if pd.notna(row.get(c))}

df_trips_dataset["GT_SET"] = df_trips_dataset.apply(build_gt_set, axis=1)
# Add ability to filter if we have out of scope labels
labels = {"PLANE", "OTHER", "ELECT_BIKE", "ELECT_SCOOTER"}
df_trips_dataset["OOS"] = df_trips_dataset["GT_SET"].apply(lambda s: isinstance(s, set) and bool(s & labels))

# Merge the two together to have Predicted and GroundTruth sets.
filter_ids = df_full_segmented_results_sets['TRIP_KEY'].unique()
df_trips_dataset_filtered = df_trips_dataset.loc[df_trips_dataset["KEY"].isin(filter_ids), ["KEY", "GT_SET", "OOS"]]
df_trips_dataset_filtered.rename(columns={"KEY": "TRIP_KEY"}, inplace=True)

df_full_segmented_results_sets = df_full_segmented_results_sets.merge(df_trips_dataset_filtered, on=["TRIP_KEY"], how="left", validate="one_to_one")

# -----------------JACUARD CALCULATIONS AND APPLY TO DS -----------------------

df_full_segmented_results_sets["JACCARD"] = df_full_segmented_results_sets.apply(lambda r: jaccard_exclude_extra_walking(r["PRED_SET"], r["GT_SET"]), axis=1)

In [29]:
# Summary stats
print("Full DS Jaccard mean:")
print(df_full_segmented_results_sets['JACCARD'].mean())

print("Without labels like Plane and Other DS Jaccard mean:")
print(df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]['JACCARD'].mean())

# Calculate percentage of exact matches
exact_match_pct = len(df_full_segmented_results_sets[df_full_segmented_results_sets['JACCARD'] == 1]) / len(df_full_segmented_results_sets['JACCARD'])
print(f"Exact match (%): {exact_match_pct*100}")

print("Exact match removing modes like plane, elect modes...")
filt = df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]
em_pct_filtered = len(filt[filt['JACCARD'] == 1]) / len(filt['JACCARD'])
print(f"Exact match (%): {em_pct_filtered*100}")

# Save out to file.
df_full_segmented_results_sets.to_csv("/home/s232534/scripts/segmentation_pelt/multimodal_classification/multi-modal-classification-RESULTS-SPEED-BR.csv")

Full DS Jaccard mean:
0.47625351503352803
Without labels like Plane and Other DS Jaccard mean:
0.48246608850344674
Exact match (%): 17.427978198806123
Exact match removing modes like plane, elect modes...
Exact match (%): 17.918612408272182


To evaluate dominant-mode recovery, segment-level predictions were aggregated to the trip level using a weighted voting scheme. For each segment, a dominance score was computed as the product of the number of GPS observations in the segment and the classifier’s predicted confidence for the assigned mode. These scores were summed across all segments belonging to the same trip and predicted mode. The dominant predicted mode for a trip was then defined as the mode with the highest cumulative score. Main-mode accuracy was computed as the proportion of trips for which the predicted dominant mode matched the annotated Main Mode.

In [30]:
results = compute_main_mode_accuracies(df_full_segmented_results)
print("MainModeAccuracy (conf only):", results["acc_conf_only"])
print("MainModeAccuracy (points * conf):", results["acc_points_x_conf"])

MainModeAccuracy (conf only): 0.2777056838826888
MainModeAccuracy (points * conf): 0.32533091097845834


### Addition of Route-Matched signals segmentation model

In [31]:
# First we must build up a complete segment feature table 
# as we have previously done with the non-segmented trips

# For posterity we reload the data:
df_motion_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_motion_segmented_SIGNAL.parquet") 
df_motion_percentile_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/percentile_map_segmented_SIGNAL.parquet") # _map means it still contains many percentile features that needs to be "optimised"
df_stops_map = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_stop_segmented_SIGNAL.parquet")
df_route_match_features = pd.read_parquet("/home/s232534/scripts/segmentation_pelt/multimodal_classification/Features/features_route_match_segmented_SIGNAL.parquet")

# filter for multi-modal
df_motion_features = df_motion_features.loc[df_motion_features['MULTI_MODAL']] 
df_motion_percentile_map = df_motion_percentile_map.loc[df_motion_percentile_map['MULTI_MODAL']] 
df_stops_map = df_stops_map.loc[df_stops_map['MULTI_MODAL']] 
df_route_match_features = df_route_match_features.loc[df_route_match_features['MULTI_MODAL']] 

# Drop modes we will not be using for this thesis and modes to merge and remap:
modes_to_drop = ['OTHER', 'PLANE', 'ON_DEMAND', 'ELECT_BIKE', 'ELECT_SCOOTER', 'TWO_WHEELER']  
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN'
}

threshold: dict = { 
            "WALKING": 6.5, # under 25 km/h
            "BIKE": 10, # 36 km/h
            "CAR": 39,
            "BUS": 14,  # ~50 km/h      
            "TRAMWAY": 19.5,     # ~70 km/h
            "SUBWAY": 11.1, # 40 km/h
            "TRAIN": 61,  # ~ 220 km/h     
        }

df_motion_features = functions.remap_and_drop_modes_df(df_motion_features, mode_remap, modes_to_drop)
df_motion_percentile_map = functions.remap_and_drop_modes_df(df_motion_percentile_map, mode_remap, modes_to_drop)
df_stops_map = functions.remap_and_drop_modes_df(df_stops_map, mode_remap, modes_to_drop)
df_route_match_features = functions.remap_and_drop_modes_df(df_route_match_features, mode_remap, modes_to_drop)

# Filter journeys that are short... they will not provide enough data to accurately classify them.
# We can use our non-segmented feature table to extract the total times.
df_motion_features_nonsegmented = pd.read_csv("/home/s232534/data/stats_071125/statistical_summary_071125.csv") 
df_motion_features_nonsegmented = df_motion_features_nonsegmented.loc[df_motion_features_nonsegmented['MULTI_MODAL']] 

#df_motion_features_nonsegmented = functions.filter_motion_data_speed_avg(df_motion_features_nonsegmented, threshold) We can't really filter by mean because we don't know what modes are exactly present
df_motion_features_nonsegmented = functions.filter_temporal_data(df_motion_features_nonsegmented, time_thres=180)

# We can propogate this filtering by taking the remaining IDs as a mask essentially
keep_ids = set(df_motion_features_nonsegmented['TRIP_KEY'])

df_motion_features = df_motion_features[df_motion_features['TRIP_KEY'].isin(keep_ids)].copy()
df_motion_percentile_map = df_motion_percentile_map[df_motion_percentile_map['TRIP_KEY'].isin(keep_ids)].copy()
df_stops_map = df_stops_map[df_stops_map['TRIP_KEY'].isin(keep_ids)].copy()
df_route_match_features = df_route_match_features[df_route_match_features['KEY'].isin(keep_ids)].copy()

# Intermediatory validation (should be True True):
t1: bool = (len(df_motion_features) == len(df_motion_percentile_map) == len(df_stops_map) == len(df_route_match_features))
t2: bool = (sorted(df_motion_features['TRIP_KEY']) == sorted(df_motion_percentile_map['TRIP_KEY']) == sorted(df_stops_map['TRIP_KEY']) == sorted(df_route_match_features['KEY']))
assert t1 and t2, f"Validation FAILED: length={t1}, keys={t2}"

# Finally we need to extract our optimal percentiles from the maps. Since we are classifying under the guise of single-modal journeys
# we can just re-use our single-modal optimal percentiles we calculated for training the single-modal classifier.
# this is contained in the 'percentile_optimal_list' and 'percentile_optimal_list_stop' found in the earlier cells.
# NOTE: We need to remove the features cols we do not use and are not generated upstream in these new scripts. (BC_ and BR_ non-WGS84)
# NOTE: IF re-running only this segment - remember to re-run the single-mode classifier data loading and filtering at a minimum.

percentile_optimal_list_segment = percentile_optimal_list.copy() # safer to copy
percentile_optimal_list_stop_segment = percentile_optimal_list_stop.copy()

percentile_optimal_list_segment.append('SEGMENT')
percentile_optimal_list_stop_segment.append('SEGMENT') # <- Need to add the segment col too so that we don't lose it as we are no linking on ID and SEGMENT

drop = {"BC_P50", "BR_P50"}
percentile_optimal_list_segment = [c for c in percentile_optimal_list_segment if c not in drop]

df_optimal_motion_features = df_motion_percentile_map[percentile_optimal_list_segment]
df_optimal_stop_features = df_stops_map[percentile_optimal_list_stop_segment]

# Rebuild the stops
# Find all _Pxx stops we remove then -> re-introduce the optimal ones again.
pattern = re.compile(r'_P(\d+)$')
drop_cols_Pxx = [
    col for col in df_stops_map.columns
    if (match := pattern.search(col))
]
drop_cols_Pxx.extend(['MAIN_MODE', 'ID', 'FILE_NAME', 'MULTI_MODAL', 'MAIN_MODE_ORIG']) # <- Remove additional cols from this one too otherwise we double up.
df_stops_dropped = df_stops_map.drop(columns=drop_cols_Pxx)
df_stops_features = df_optimal_stop_features.merge(df_stops_dropped, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Rebuild the full motion features including the optimal percentiles:
drop_motion_cols = ['SPEED_85P', 'ACC_85P', 'JERK_85P', 'B_85P_WGS84', 'BC_85P_WGS84', 'BR_85P_WGS84', 'MULTI_MODAL', 'FILE_NAME', 'MAIN_MODE_ORIG', 'ID', 'B_AVG_WGS84', 'B_MED_WGS84', 'B_STD_WGS84']
df_motion_features = df_motion_features.drop(columns=drop_motion_cols)
df_motion_features = df_motion_features.merge(df_optimal_motion_features, on=['TRIP_KEY', 'SEGMENT'], how='left', validate='one_to_one')

# Fix up route matching dataset to be ready to merge in
df_route_match_features.rename(columns={"KEY": "TRIP_KEY"}, inplace=True) # Should be fixed upstream but the name does not align with the standard.
df_route_match_features = df_route_match_features.drop(columns=['MAIN_MODE', 'MAIN_MODE_ORIG', 'MULTI_MODAL'])
cols_rm = ['bus_match_pct', 'tram_match_pct', 'train_match_pct', 'subway_match_pct']
df_route_match_features[cols_rm] = df_route_match_features[cols_rm].fillna(0)

# Build up the final dataframe ready for modelling:
df_full_segmented_features = (
    df_motion_features
      .merge(df_stops_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
      .merge(df_route_match_features, on=["TRIP_KEY", "SEGMENT"], how="left", validate="one_to_one")
)

Filtered by TOTAL_TIME > 180. Dropped 120 rows.


In [38]:
df_full_segmented_features.keys()

Index(['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'TOTAL_TIME', 'AVG_PT_DIST_WGS84',
       'MED_PT_DIST_WGS84', 'DIST_TOTAL_WGS84', 'GPS_POINTS', 'SPEED_AVG',
       'SPEED_MED', 'SPEED_STD', 'ACC_AVG', 'ACC_MED', 'ACC_STD', 'JERK_AVG',
       'JERK_MED', 'JERK_STD', 'BC_AVG_WGS84', 'BC_MED_WGS84', 'BC_STD_WGS84',
       'BR_AVG_WGS84', 'BR_MED_WGS84', 'BR_STD_WGS84', 'ACC_P95',
       'BC_WGS84_P50', 'BR_WGS84_P50', 'JERK_P95', 'SPEED_P95',
       'STOPS_CLOSEST_BUS_P95', 'STOPS_CLOSEST_METRO_P95',
       'STOPS_CLOSEST_RAIL_P20', 'STOPS_CLOSEST_TRAM_P15', 'STOPS_MINDIST_P95',
       'STOPS_NUM_P5', 'STOPS_NUM_AVG', 'STOPS_NUM_MAX', 'STOPS_NUM_MIN',
       'STOPS_MINDIST_AVG', 'DURATION_TRAM', 'DURATION_METRO', 'DURATION_RAIL',
       'DURATION_BUS', 'MIN_DIST_TRAM', 'MIN_DIST_METRO', 'MIN_DIST_RAIL',
       'MIN_DIST_BUS', 'bus_match_pct', 'tram_match_pct', 'train_match_pct',
       'subway_match_pct'],
      dtype='object')

In [33]:
# Just for clarity we reload the classifer that was saved earlier in the book
xgb_single_classifier = XGBClassifier()
xgb_single_classifier.load_model("/home/s232534/scripts/segmentation_pelt/multimodal_classification/xgb_singlemodal_classifier.ubj")
le = joblib.load("/home/s232534/scripts/segmentation_pelt/multimodal_classification/label_encoder_single.joblib")

# Our feature set
X = df_full_segmented_features.copy()
X = X.drop(columns=['TRIP_KEY', 'SEGMENT', 'MAIN_MODE'])

# Run prediction:
proba = xgb_single_classifier.predict_proba(X) 
idx = proba.argmax(axis=1)

# Fill df with results:
df_full_segmented_results = df_full_segmented_features[['TRIP_KEY', 'SEGMENT', 'MAIN_MODE', 'GPS_POINTS']].copy()
 
df_full_segmented_results["PRED_MODE"] = le.inverse_transform(idx)
df_full_segmented_results["PRED_CONF"] = proba.max(axis=1)

# -----------BATCHING AND CREATING GROUND TRUTHS ---------

# Batch the results into sets
df_full_segmented_results_sets = (
    df_full_segmented_results.groupby("TRIP_KEY")["PRED_MODE"]
    .agg(lambda s: set(s.dropna()))
    .reset_index(name="PRED_SET")
)

# Create ground truth sets:
df_trips_dataset = pd.read_csv("/home/lauale/netmob2025/data/trips_dataset.csv")
bad_key = "2_1946-monday-44935-1" # Duplicate key that actually stems from the original dataset we need to drop it.
df_trips_dataset = df_trips_dataset.loc[df_trips_dataset["KEY"] != bad_key].copy()

# Before generating the sets we need to remap some names:
# We also for now keep ELECT_BIKE and ELECT_SCOOTER and remap to BIKE
mode_remap = {
    'PRIV_CAR_DRIVER': 'CAR',
    'PRIV_CAR_PASSENGER': 'CAR',
    'LIGHT_COMM_VEHICLE': 'CAR',
    'TAXI': 'CAR',
    'TRAIN_EXPRESS': 'TRAIN',
    # 'ELECT_BIKE': 'BIKE',
    # 'ELECT_SCOOTER': 'BIKE',
}
mode_cols = ["Main_Mode", "Mode_1", "Mode_2", "Mode_3", "Mode_4", "Mode_5"]
df_trips_dataset[mode_cols] = df_trips_dataset[mode_cols].replace(mode_remap)

# Build the ground truths
def build_gt_set(row):
    return {row[c] for c in mode_cols if pd.notna(row.get(c))}

df_trips_dataset["GT_SET"] = df_trips_dataset.apply(build_gt_set, axis=1)
# Add ability to filter if we have out of scope labels
labels = {"PLANE", "OTHER", "ELECT_BIKE", "ELECT_SCOOTER"}
df_trips_dataset["OOS"] = df_trips_dataset["GT_SET"].apply(lambda s: isinstance(s, set) and bool(s & labels))

# Merge the two together to have Predicted and GroundTruth sets.
filter_ids = df_full_segmented_results_sets['TRIP_KEY'].unique()
df_trips_dataset_filtered = df_trips_dataset.loc[df_trips_dataset["KEY"].isin(filter_ids), ["KEY", "GT_SET", "OOS"]]
df_trips_dataset_filtered.rename(columns={"KEY": "TRIP_KEY"}, inplace=True)

df_full_segmented_results_sets = df_full_segmented_results_sets.merge(df_trips_dataset_filtered, on=["TRIP_KEY"], how="left", validate="one_to_one")

# -----------------JACUARD CALCULATIONS AND APPLY TO DS -----------------------

df_full_segmented_results_sets["JACCARD"] = df_full_segmented_results_sets.apply(lambda r: jaccard_exclude_extra_walking(r["PRED_SET"], r["GT_SET"]), axis=1)

In [34]:
# Summary stats
print("Full DS Jaccard mean:")
print(df_full_segmented_results_sets['JACCARD'].mean())

print("Without labels like Plane and Other DS Jaccard mean:")
print(df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]['JACCARD'].mean())

# Calculate percentage of exact matches
exact_match_pct = len(df_full_segmented_results_sets[df_full_segmented_results_sets['JACCARD'] == 1]) / len(df_full_segmented_results_sets['JACCARD'])
print(f"Exact match (%): {exact_match_pct*100}")

print("Exact match removing modes like plane, elect modes...")
filt = df_full_segmented_results_sets[df_full_segmented_results_sets['OOS'] == False]
em_pct_filtered = len(filt[filt['JACCARD'] == 1]) / len(filt['JACCARD'])
print(f"Exact match (%): {em_pct_filtered*100}")

# Save out to file.
df_full_segmented_results_sets.to_csv("/home/s232534/scripts/segmentation_pelt/multimodal_classification/multi-modal-classification-RESULTS-ROUTE-SIGNAL.csv")

Full DS Jaccard mean:
0.495771144278607
Without labels like Plane and Other DS Jaccard mean:
0.5024438514565266
Exact match (%): 18.258499870230988
Exact match removing modes like plane, elect modes...
Exact match (%): 18.77251501000667


In [35]:
results = compute_main_mode_accuracies(df_full_segmented_results)
print("MainModeAccuracy (conf only):", results["acc_conf_only"])
print("MainModeAccuracy (points * conf):", results["acc_points_x_conf"])

MainModeAccuracy (conf only): 0.2420192058136517
MainModeAccuracy (points * conf): 0.3044381001816766
